# Análisis de Hipótesis — Taco Bueno Pricing (H1–H8)

Notebook de **exploración** sobre las tablas Gold (`workspace.default.taco_bueno_gold_*`) generado por el pipeline [05_Pipeline_Job](#notebook-1988745176962912). Aquí se juega con los resultados y se validan las hipótesis del equipo, separado del pipeline productivo (03 Silver / 04 Gold).

Cada celda evalúa una hipótesis usando las tablas Gold (H7 requiere granularidad de orden y se calcula desde la Silver). El veredicto es **evidencia descriptiva**: confirma o refuta la consistencia con los datos, NO causalidad — para afirmar causa se necesita un experimento A/B o grupo de control.

Se pueden añadir celdas libremente: ninguna modifica tablas (solo lectura), así que este notebook nunca rompe el pipeline.

## Resumen de hipótesis

| # | Hipótesis | Estrategia propuesta | Fuente |
|---|---|---|---|
| H1 | Drive Thru no necesita descuentos | Mantenerlo sin promoción | gold_channel |
| H2 | Steak Soft Taco canibalizó su precio ($429K) | Revisar su descuento | gold_product |
| H3 | Recortar descuentos no causó la caída de tráfico | Buscar otras causas | gold_monthly |
| H4 | Oportunidad happy hour 3–6pm | Promociones en esa ventana | gold_hourly |
| H5 | App: 50.9% con descuento en solo 7,486 transacciones | Repensar enfoque digital | gold_channel |
| H6 | Promos de fin de semana emparejan tráfico | Family meals y bundles | gold_dayofweek |
| H7 | Descuentos en órdenes pequeñas no recuperan revenue | Limitarlos | Silver (orden) |
| H8 | Descuentos en sides/condimentos son regalos | Dejar de descontarlos | gold_product |

In [0]:
# H1 — "Drive Thru does not need discounts" → Estrategia: mantenerlo promotion-free
h1 = spark.sql("""
SELECT channel, transactions, orders, revenue,
       discount_rate_pct, avg_ticket_discounted, avg_ticket_non_discounted,
       ROUND(avg_ticket_non_discounted - avg_ticket_discounted, 2) AS ticket_gap
FROM workspace.default.taco_bueno_gold_channel
ORDER BY transactions DESC
""").toPandas()
display(h1)

dt = h1.loc[h1['channel'] == 'Drive Thru'].iloc[0]
mean_rate = h1['discount_rate_pct'].mean()
print(f"\nH1 — Evidencia:")
print(f"• Drive Thru: {dt['transactions']:,} transacciones ({dt['transactions']/h1['transactions'].sum()*100:.1f}% del total), tasa de descuento {dt['discount_rate_pct']}% (promedio de canales: {mean_rate:.1f}%).")
print(f"• Ticket sin descuento ${dt['avg_ticket_non_discounted']} vs con descuento ${dt['avg_ticket_discounted']} (gap ${dt['ticket_gap']}).")
veredicto = "CONSISTENTE" if dt['discount_rate_pct'] < mean_rate else "NO CONSISTENTE"
print(f"VEREDICTO: {veredicto} con H1 — Drive Thru ya opera por debajo de la media de descuentos y su ticket sin descuento es alto; no muestra dependencia del descuento. (Descriptivo, no causal)")

In [0]:
# H2 — "Steak Soft Taco canibalizó su propio precio ($429K perdidos)" → Revisar su descuento
h2 = spark.sql("""
SELECT menu_item, report_group, transactions, units_sold, revenue,
       discount_rate_pct, discount_amount, discount_per_unit,
       avg_ticket_discounted, avg_ticket_non_discounted
FROM workspace.default.taco_bueno_gold_product
WHERE menu_item = 'Steak Soft Taco'
""").toPandas()
top_disc = spark.sql("""
SELECT menu_item, transactions, discount_amount, discount_rate_pct
FROM workspace.default.taco_bueno_gold_product
ORDER BY discount_amount DESC LIMIT 5
""").toPandas()
display(h2)
display(top_disc)

r = h2.iloc[0]
print(f"\nH2 — Evidencia:")
print(f"• Steak Soft Taco: {r['transactions']:,} transacciones, {r['units_sold']:,} unidades, revenue ${r['revenue']:,.0f}.")
print(f"• Tasa de descuento {r['discount_rate_pct']}% — la tasa de descuento más alta del menú; por monto total descontado aparece en el Top 5 (encabezado por Beef Crispy Taco, ${top_disc.iloc[0]['discount_amount']:,.0f}).")
print(f"• ${r['discount_amount']:,.2f} descontados = ${r['discount_per_unit']:.2f} por unidad vendida ({r['discount_amount']/r['revenue']*100:.1f}% de su revenue).")
print(f"• Ticket con descuento ${r['avg_ticket_discounted']} vs sin descuento ${r['avg_ticket_non_discounted']}.")
print("VEREDICTO: CONSISTENTE con H2 — la magnitud del descuento justifica revisar precio/eligibilidad. La canibalización estricta (si el descuento quita venta a otros tacos) requiere análisis de canastas — ver próximos pasos.")

In [0]:
# H3 — "Cutting discounts did not cause the traffic drop" → Buscar otras causas
h3 = spark.sql("""
SELECT month, transactions, orders, active_stores, discount_rate_pct,
       ROUND(orders / active_stores, 1) AS orders_per_store
FROM workspace.default.taco_bueno_gold_monthly
WHERE month >= '2025-02-01' AND month < '2026-09-01'  -- excluye meses parciales (ene-25, sep-26)
ORDER BY month
""").toPandas()
h3['mes'] = h3['month'].dt.strftime('%Y-%m')
display(h3[['mes', 'transactions', 'orders_per_store', 'discount_rate_pct']])

corr_trans = h3['transactions'].corr(h3['discount_rate_pct'])
corr_store = h3['orders_per_store'].corr(h3['discount_rate_pct'])
h3['anio'] = h3['month'].dt.year
avg_25 = h3.loc[h3['anio'] == 2025, 'transactions'].mean()
avg_26 = h3.loc[h3['anio'] == 2026, 'transactions'].mean()
print(f"\nH3 — Evidencia (feb-2025 a ago-2026, meses completos):")
print(f"• Correlación mensual: tasa de descuento vs transacciones = {corr_trans:.2f}; vs órdenes por tienda = {corr_store:.2f}.")
print(f"• Transacciones promedio: 2025 = {avg_25/1e6:.2f}M/mes vs 2026 = {avg_26/1e6:.2f}M/mes ({(avg_26-avg_25)/avg_25*100:+.1f}%).")
if corr_trans < -0.5:
    v = "Los meses con MÁS descuento tuvieron MENOS transacciones — la evidencia apoya que el recorte no causó la caída."
elif corr_trans > 0.5:
    v = "Los meses con más descuento tuvieron más transacciones — NO se puede descartar que el recorte afectara el tráfico; requiere análisis por cohortes y estacionalidad."
else:
    v = "Correlación débil — el recorte de descuentos no explica por sí solo el tráfico; buscar otras causas (estacionalidad, aperturas/cierres, competencia)."
print(f"VEREDICTO: {v} (Correlación ≠ causalidad)")

In [0]:
# H4 — "There is a happy hour opportunity from 3 to 6 pm" → Promociones en esa ventana
h4 = spark.sql("""
SELECT hour_of_day, transactions, revenue, avg_ticket, discount_rate_pct,
       ROUND(revenue / SUM(revenue) OVER (), 4) AS revenue_share
FROM workspace.default.taco_bueno_gold_hourly
ORDER BY hour_of_day
""").toPandas()
display(h4)

tarde = h4[h4['hour_of_day'].isin([15, 16, 17])]
pico = h4.loc[h4['revenue'].idxmax()]
print(f"\nH4 — Evidencia:")
print(f"• Ventana 3–6pm (15–17h): {tarde['revenue_share'].sum()*100:.1f}% del revenue, ticket promedio ${tarde['avg_ticket'].mean():.2f}.")
print(f"• Hora pico: {int(pico['hour_of_day'])}h (${pico['revenue']/1e6:.1f}M).")
print(f"• Tasa de descuento 15–17h: {tarde['discount_rate_pct'].mean():.2f}% vs media diaria {h4['discount_rate_pct'].mean():.2f}%.")
media_desc = h4['discount_rate_pct'].mean()
comparacion = "por ENCIMA de la media" if tarde['discount_rate_pct'].mean() > media_desc else "por debajo de la media"
print(f"VEREDICTO: EVALUAR — la franja 15–17h genera {tarde['revenue_share'].sum()*100:.0f}% del revenue con descuentos {comparacion} ({tarde['discount_rate_pct'].mean():.2f}% vs {media_desc:.2f}%): ya recibe descuentos, así que un happy hour debe rediseñarse (menos descuento general, oferta específica de la franja) y validarse con A/B.")

In [0]:
# H5 — "App discounts 50.9% with only 7,486 transactions" → Rethink the digital approach
h5 = spark.sql("""
SELECT channel, transactions, orders, revenue, discount_rate_pct,
       avg_ticket_discounted, avg_ticket_non_discounted, discount_amount
FROM workspace.default.taco_bueno_gold_channel
WHERE channel IN ('Mobile', 'Online', 'Online: DoorDash', 'Online: UberEats', 'Online: GrubHub', 'Online: Zipline')
ORDER BY transactions DESC
""").toPandas()
display(h5)

mob = h5[h5['channel'] == 'Mobile']
total_trans = h5['transactions'].sum()
if len(mob):
    m = mob.iloc[0]
    print(f"\nH5 — Evidencia:")
    print(f"• Mobile: {m['transactions']:,} transacciones ({m['transactions']/total_trans*100:.2f}% del digital; {m['transactions']/64862789*100:.3f}% del total), tasa de descuento {m['discount_rate_pct']}%.")
    print(f"• Digital agregado (Mobile + Online*): {total_trans:,} transacciones, ${h5['discount_amount'].sum():,.0f} descontados.")
    print(f"• Ticket Mobile con descuento ${m['avg_ticket_discounted']} vs sin ${m['avg_ticket_non_discounted']}.")
    veredicto = "CONSISTENTE" if m['discount_rate_pct'] > 30 else "REVISAR"
    print(f"VEREDICTO: {veredicto} con H5 — el volumen de app es marginal; su tasa de descuento elevada sugiere reorientar el incentivo digital a captación de primeras órdenes con cupones únicos medibles, en vez de descuento recurrente.")

In [0]:
# H6 — "Weekend promos even out traffic" → Family meals and bundles
h6 = spark.sql("""
SELECT day_of_week_num, day_name, is_weekend, store_days, transactions, orders,
       avg_orders_per_store_day, revenue, discount_rate_pct
FROM workspace.default.taco_bueno_gold_dayofweek
ORDER BY day_of_week_num
""").toPandas()
display(h6)

wknd = h6[h6['is_weekend']]
lab = h6[~h6['is_weekend']]
debil = h6.loc[h6['avg_orders_per_store_day'].idxmin()]
print(f"\nH6 — Evidencia:")
print(f"• Fin de semana: {wknd['avg_orders_per_store_day'].mean():.1f} órdenes/tienda/día vs laboral {lab['avg_orders_per_store_day'].mean():.1f} ({(wknd['avg_orders_per_store_day'].mean()-lab['avg_orders_per_store_day'].mean())/lab['avg_orders_per_store_day'].mean()*100:+.1f}%).")
print(f"• Día más débil: {debil['day_name']} ({debil['avg_orders_per_store_day']} órdenes/tienda/día).")
print(f"• Tasa de descuento finde {wknd['discount_rate_pct'].mean():.2f}% vs laboral {lab['discount_rate_pct'].mean():.2f}%.")
print(f"VEREDICTO: EVALUAR — un bundle/family meal dirigido al día más débil ({debil['day_name']}) tiene base de datos para 'emparejar' tráfico. Medir con grupo de control antes de generalizar la promo.")

In [0]:
# H7 — "Discounts on small orders do not make up for lost revenue" → Limitarlos
# Grain de ORDEN (store_id + check_number). Nota: dado que `total` parece repetirse a nivel
# check por línea, el monto de la orden se aproxima con MAX(total) y MAX(total_discount).
h7 = spark.sql("""
WITH ordenes AS (
  SELECT store_id, check_number,
         MAX(total) AS order_total,
         MAX(total_discount) AS order_discount,
         MAX(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) AS has_discount,
         COUNT(*) AS lines
  FROM workspace.default.taco_bueno_silver
  WHERE transaction_type != 'Void'
  GROUP BY store_id, check_number
)
SELECT CASE WHEN order_total < 10 THEN '1. <$10'
            WHEN order_total < 20 THEN '2. $10–19.99'
            WHEN order_total < 30 THEN '3. $20–29.99'
            WHEN order_total < 50 THEN '4. $30–49.99'
            ELSE '5. $50+' END AS order_bucket,
       COUNT(*) AS orders,
       ROUND(AVG(order_total), 2) AS avg_order_value,
       ROUND(SUM(has_discount) * 100.0 / COUNT(*), 2) AS pct_orders_with_discount,
       ROUND(AVG(CASE WHEN has_discount = 1 THEN order_discount END), 2) AS avg_discount_when_given,
       ROUND(SUM(CASE WHEN has_discount = 1 THEN order_discount ELSE 0 END), 0) AS discount_given_total
FROM ordenes
GROUP BY 1 ORDER BY 1
""").toPandas()
display(h7)

tot_desc = h7['discount_given_total'].sum()
small = h7[h7['order_bucket'] == '1. <$10'].iloc[0]
print(f"\nH7 — Evidencia:")
print(f"• Órdenes <$10: {small['orders']:,} ({small['orders']/h7['orders'].sum()*100:.1f}% de las órdenes), {small['pct_orders_with_discount']}% reciben descuento, ticket promedio ${small['avg_order_value']}.")
print(f"• Descuento dado en órdenes <$10: ${small['discount_given_total']:,.0f} ({small['discount_given_total']/tot_desc*100:.1f}% del descuento total).")
print("VEREDICTO: EVALUAR — si el bucket pequeño concentra un % de descuento desproporcionado a su revenue, definir un umbral mínimo de compra (p. ej. $10) y validarlo con A/B antes de limitar.")

In [0]:
# H8 — "Discounts on sides and condiments are pure giveaways" → Stop discounting them
h8 = spark.sql("""
SELECT report_group,
       COUNT(*) AS products,
       SUM(transactions) AS transactions,
       ROUND(SUM(revenue), 0) AS revenue,
       ROUND(SUM(discount_amount), 0) AS discount_total,
       ROUND(SUM(discounted_transactions) * 100.0 / SUM(transactions), 2) AS discount_rate_pct,
       ROUND(SUM(discount_amount) / NULLIF(SUM(revenue), 0) * 100, 2) AS discount_as_pct_of_revenue
FROM workspace.default.taco_bueno_gold_product
WHERE report_group LIKE '%Condiments%' OR report_group LIKE '%Sides%'
   OR report_group LIKE '%(I)%' OR report_group LIKE '%Substitution%'
GROUP BY report_group
ORDER BY discount_total DESC
""").toPandas()
display(h8)

tot8 = h8['discount_total'].sum()
print(f"\nH8 — Evidencia:")
for _, r in h8.iterrows():
    print(f"• {r['report_group']}: {r['transactions']:,} transacciones, ${r['discount_total']:,.0f} descontados ({r['discount_rate_pct']}% de líneas con descuento; {r['discount_as_pct_of_revenue']}% del revenue del grupo).")
print(f"• Total descontado en sides/condimentos/ítems no-comida: ${tot8:,.0f}.")
print("VEREDICTO: EVALUAR — son 'giveaways' si el descuento no genera unidades incrementales; recomendar restringir item_discount en estos grupos a combos y medir canasta incremental.")